# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a Croissant schema and accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the required mlcroissant package (restart the kernel if upgrading)
!pip install --quiet mlcroissant

## 1. Data Loading

We use `mlcroissant` to load the dataset's Croissant schema and metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their `@id`s, and fields. Record sets structure the data—each corresponds to a table.

In [ ]:
# List all record sets with their @id and fields
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# For each record set, print the field @id's
print("\nFields in Each Record Set:")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    if 'field' in rs and isinstance(rs['field'], list):
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"  - Field @id: {field.get('@id', '[no id]')}, name: {field.get('name', '[no name]')}")
            else:
                print(f"  - Field @id: {field}")
    print()

## 3. Data Extraction

Here we select a record set and load its data into a DataFrame for analysis.

> **All references are by their `@id`.**

In [ ]:
# Determine the main data record set
# We'll select the first available record set for demonstration
main_record_set_id = record_sets[0]['@id'] if record_sets else None
print(f"Loading records from record set @id: {main_record_set_id}")

dataframes = {}
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df_main = pd.DataFrame(records)
    dataframes[main_record_set_id] = df_main
    print(f"Columns in main record set (@id={main_record_set_id}):")
    print(df_main.columns.tolist())
    display(df_main.head())
else:
    print("No record sets found in schema!")

## 4. Exploratory Data Analysis (EDA)

We perform basic EDA: select a numeric field (by its `@id`/column name), filter records, normalize, and group by another field.

The available column names are printed above. Please select appropriate `@id`s/column names for numeric and grouping fields as shown below.

In [ ]:
# Example: Choose a numeric field (column name) and a grouping field (column name or @id)
numeric_field = None
group_field = None

# Recommend a numeric field by checking dataframe dtypes
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print("Numeric candidate fields:")
    print(df.select_dtypes(include=['number']).columns.tolist())
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    # If we find a numeric field, use it
    if numeric_candidates:
        numeric_field = numeric_candidates[0]

    # Suggest group-able fields (e.g. categorical/text fields with few unique values)
    group_candidates = [col for col in df.columns if df[col].dtype == "object" and df[col].nunique() < 10]
    if group_candidates:
        group_field = group_candidates[0]

    # If we found a numeric field, proceed with EDA
    if numeric_field:
        print(f"\nUsing numeric field: '{numeric_field}'")
        if group_field:
            print(f"Using grouping field: '{group_field}'")

        # Filter above a threshold (median for demo)
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records where {numeric_field} > {threshold} (median): {len(filtered_df)} row(s)")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by group_field if chosen
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field}:")
            display(grouped_df)
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("Dataframe for main record set is missing.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and if grouped data is available, also show a grouped bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if numeric field is extracted
if main_record_set_id and main_record_set_id in dataframes and numeric_field:
    df = dataframes[main_record_set_id]

    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouped_df exists (see EDA step), show bar plot
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(6,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion

- We loaded the FAIR² colorectal cancer dataset Croissant schema, reviewed available record sets, and loaded tabular data for analysis.
- We demonstrated EDA by filtering and normalizing a numeric field and grouping records by a categorical field.
- Data distributions and grouped values were visualized, highlighting possible cohort patterns.

> Continue your exploration by inspecting more record sets and fields using their `@id` values with `mlcroissant`!